# Hansen《Econometrics》第 2 章习题解答

**Chapter 2 Conditional Expectation and Projection**

对应书稿 PDF 第 80–81 页（印刷页 60–61），§2.34 Exercises。

完整推导见同目录 `Hansen_Ch02_Exercises_Solutions.md`。本 notebook 做 **Exercise 2.4、2.16** 数值验证，并汇总其余题目答案。


## Exercise 2.1–2.3

**2.1** 由迭代期望：
$$E[E[E[Y\mid X_1,X_2,X_3]\mid X_1,X_2]\mid X_1]=E[Y\mid X_1].$$

**2.2** 若 $E[Y\mid X]=a+bX$，则
$$E[YX]=aE[X]+bE[X^2].$$

**2.3** Theorem 2.4.4：
$$E[h(X)e]=E[h(X)E[e\mid X]]=0.$$


## Exercise 2.4

联合分布：

|  | $X=0$ | $X=1$ |
|--|------:|------:|
| $Y=0$ | 0.1 | 0.2 |
| $Y=1$ | 0.4 | 0.3 |


In [ ]:
import numpy as np
import pandas as pd

# joint P(Y,X): rows Y=0,1; cols X=0,1
P = np.array([[0.1, 0.2],
              [0.4, 0.3]])
assert np.isclose(P.sum(), 1.0)

px = P.sum(axis=0)
print("P(X) =", px)

rows = []
for x in [0, 1]:
    py_x = P[:, x] / px[x]
    ey = float(py_x[1])           # E[Y|X] since Y in {0,1}
    ey2 = ey                      # Y^2 = Y
    var = ey2 - ey**2
    rows.append({"X": x, "E[Y|X]": ey, "E[Y^2|X]": ey2, "Var[Y|X]": var})
    print(f"X={x}: E[Y|X]={ey:.4f}, E[Y^2|X]={ey2:.4f}, Var={var:.4f}")

pd.DataFrame(rows)


## Exercise 2.5–2.9 要点

- **2.5** $\sigma^2(X)=E[e^2\mid X]$ 是 $e^2$ 的 CEF，故为最佳均方预测。
- **2.6** $m(X)\perp e$ $\Rightarrow$ $\mathrm{Var}(Y)=\mathrm{Var}(m(X))+\sigma^2$。
- **2.7** $\sigma^2(X)=E[Y^2\mid X]-(E[Y\mid X])^2$。
- **2.8** Poisson：$E[Y\mid X]=\mathrm{Var}[Y\mid X]=X'\beta$；CEF 线性成立，但一般 **异方差**。
- **2.9** 饱和模型：截距 $+X_1+$ 类别虚拟 $+$ 交互（6 个参数）。


## Exercise 2.10–2.14 True/False

| 题 | 答案 | 理由 |
|:--:|:----:|------|
| 2.10 | **True** | $E[e\mid X]=0$ $\Rightarrow$ $E[X^2e]=0$ (Thm 2.4.4) |
| 2.11 | **False** | $E[Xe]=0$ 推不出 $E[X^2e]=0$ |
| 2.12 | **False** | 均值独立 $\neq$ 独立 |
| 2.13 | **False** | 投影正交 $\neq$ CEF 正交 |
| 2.14 | **False** | 同方差+均值独立仍可有高阶依赖 |


## Exercise 2.15–2.16

**2.15** 截距模型 BLP：$\alpha=E[Y]$。

**2.16** $f(x,y)=\frac{3}{2}(x^2+y^2)$，$0\le x,y\le 1$。计算 BLP 与 CEF。


In [ ]:
from scipy import integrate

def f(y, x):
    return 1.5 * (x**2 + y**2)

I, _ = integrate.dblquad(lambda y, x: f(y, x), 0, 1, 0, 1)
print("integral of density =", I)

EX, _ = integrate.dblquad(lambda y, x: x * f(y, x), 0, 1, 0, 1)
EY, _ = integrate.dblquad(lambda y, x: y * f(y, x), 0, 1, 0, 1)
EX2, _ = integrate.dblquad(lambda y, x: (x**2) * f(y, x), 0, 1, 0, 1)
EXY, _ = integrate.dblquad(lambda y, x: x * y * f(y, x), 0, 1, 0, 1)
print(f"E[X]={EX:.6f}, E[Y]={EY:.6f}, E[X^2]={EX2:.6f}, E[XY]={EXY:.6f}")

varX = EX2 - EX**2
cov = EXY - EX * EY
beta = cov / varX
alpha = EY - beta * EX
print(f"BLP: alpha={alpha:.6f}, beta={beta:.6f}")
print(f"exact: beta=-15/73={-15/73:.6f}, alpha=55/73={55/73:.6f}")

def m(x):
    fx = 1.5 * (x**2 + 1.0/3.0)
    num = 1.5 * (0.5 * x**2 + 0.25)
    return num / fx

for x in [0.0, 0.5, 1.0]:
    print(f"x={x}: m(x)={m(x):.6f}, BLP={alpha + beta*x:.6f}")

print("CEF is not affine in x => different from BLP")


## Exercise 2.17–2.22 要点

- **2.17** $E[g(X,m,s)]=0$ iff $m=\mu$ and $s=\sigma^2$。
- **2.18** $X_3$ 为 $X_2$ 仿射 $\Rightarrow Q_{XX}$ 奇异；BLP 用 $(1,X_2)$。
- **2.19** $\min d(\beta)$ 得 $\beta=(E[XX'])^{-1}E[XY]$。
- **2.20** 有密度时 (2.6) 满足 (2.57)。
- **2.21** $\gamma_1=\beta_1$ iff $E[XX_2]\beta_2=0$。
- **2.22** 排除 $X_2$ 通常 **诱导异方差**：$E[u^2\mid X_1]$ 一般依赖 $X_1$。
